In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

In [3]:
from google.colab import files
uploaded = files.upload()

Saving messy_telecom_churn_dataset.csv to messy_telecom_churn_dataset.csv


In [5]:
df = pd.read_csv("messy_telecom_churn_dataset.csv")
df.head()

,CustomerID,Tenure,InternetService,Contract,MonthlyCharges,TotalCharges,PaymentMethod,Churn
0,C00001,66,Fiber Optic,Month-to-Month,140.00,9859.85,Electronic Check,Yes
1,C00002,17,Fiber Optic,One Year,92.01,1576.77,Electronic Check,Yes
2,C00003,57,Fiber Optic,Month-to-Month,105.94,5161.17,Bank Transfer,No
3,C00004,47,Fiber Optic,One Year,41.25,2183.39,Mailed Check,No
4,C00005,32,DSL,One Year,123.04,4145.68,Credit Card,No


In [6]:
print("Shape:", df.shape)
print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (5120, 8)

Missing Values:

CustomerID           0
Tenure               0
InternetService    151
Contract             0
MonthlyCharges     152
TotalCharges       156
PaymentMethod      151
Churn                0
dtype: int64

Duplicate Rows: 120


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5120 entries, 0 to 5119
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       5120 non-null   object 
 1   Tenure           5120 non-null   int64  
 2   InternetService  4969 non-null   object 
 3   Contract         5120 non-null   object 
 4   MonthlyCharges   4968 non-null   float64
 5   TotalCharges     4964 non-null   object 
 6   PaymentMethod    4969 non-null   object 
 7   Churn            5120 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 320.1+ KB


In [9]:
### Duplicate records were removed to prevent skewed churn calculations and revenue metrics.
df = df.drop_duplicates()

print("New Shape:", df.shape)

New Shape: (5000, 8)


In [10]:
print(df[['MonthlyCharges', 'TotalCharges']].head(10))
print(df[['MonthlyCharges', 'TotalCharges']].dtypes)

   MonthlyCharges TotalCharges
0          140.00      9859.85
1           92.01      1576.77
2          105.94      5161.17
3           41.25      2183.39
4          123.04      4145.68
5          113.08      4756.95
6           38.92      1101.57
7           39.15     $1350.06
8           86.20      5216.23
9           54.85      2683.35
MonthlyCharges    float64
TotalCharges       object
dtype: object


In [11]:
##Data cleaning

df['MonthlyCharges'] = (
    df['MonthlyCharges']
    .astype(str)
    .str.strip()
)

df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')

In [12]:
df['TotalCharges'] = (
    df['TotalCharges']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [13]:
print(df[['MonthlyCharges', 'TotalCharges']].dtypes)

print("\nMissing After Conversion:")
print(df[['MonthlyCharges', 'TotalCharges']].isnull().sum())

MonthlyCharges    float64
TotalCharges      float64
dtype: object

Missing After Conversion:
MonthlyCharges    150
TotalCharges      150
dtype: int64


In [14]:
df.isnull().sum()

,0
CustomerID,0
Tenure,0
InternetService,150
Contract,0
MonthlyCharges,150
TotalCharges,150
PaymentMethod,150
Churn,0


In [15]:
#filling MonthlyCharges with Median
df['MonthlyCharges'].fillna(df['MonthlyCharges'].median(), inplace=True)

/tmp/ipykernel_4570/3475198994.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['MonthlyCharges'].fillna(df['MonthlyCharges'].median(), inplace=True)


In [16]:
## TotalCharges missing, estimate as: MonthlyCharges × Tenure

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['MonthlyCharges'] * df['Tenure']
)

In [17]:
categorical_cols = ['InternetService', 'PaymentMethod']

for col in categorical_cols:
    df[col].fillna('Unknown', inplace=True)

/tmp/ipykernel_4570/568711576.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('Unknown', inplace=True)


In [18]:
df.isnull().sum()

,0
CustomerID,0
Tenure,0
InternetService,0
Contract,0
MonthlyCharges,0
TotalCharges,0
PaymentMethod,0
Churn,0


In [19]:
#Negative tenure Negative charges Unrealistically high charges
print(df.describe())

           Tenure  MonthlyCharges  TotalCharges
count  5000.00000     5000.000000   5000.000000
mean     35.89240       71.747738   2722.856324
std      21.12127       48.419385   4723.625713
min      -5.00000      -50.000000      0.000000
25%      18.00000       52.950000   1047.547500
50%      36.00000       70.210000   2194.675000
75%      54.00000       86.622500   3698.847500
max      72.00000      999.000000  99999.000000


In [20]:
##Tenure cannot be negative.
df = df[df['Tenure'] >= 0]

In [21]:
#Fixing Invalid Monthly Charges
df = df[
    (df['MonthlyCharges'] >= 0) &
    (df['MonthlyCharges'] <= 300)
]


In [22]:
#Remove unrealistic total charges

df = df[df['TotalCharges'] <= 20000]

In [24]:
#Invalid business-rule violations and extreme outliers were removed using domain thresholds to preserve analytical integrity while preventing skew in churn and revenue calculations.

df.describe()

,Tenure,MonthlyCharges,TotalCharges
count,4960.000000,4960.000000,4960.000000
mean,35.979637,70.110282,2525.599194
std,21.054290,24.406571,1830.614410
min,0.000000,15.000000,0.000000
25%,18.000000,52.965000,1045.352500
50%,36.000000,70.210000,2187.575000
75%,54.000000,86.520000,3681.057500
max,72.000000,140.000000,10341.060000


In [25]:
#Inconsistent categorical labels caused by casing, whitespace, and alternate naming conventions were standardized to ensure accurate aggregation and segmentation.

categorical_cols = ['InternetService', 'Contract', 'PaymentMethod', 'Churn']

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [26]:
df['InternetService'] = df['InternetService'].replace({
    'fiber': 'fiber optic',
    'fiber optic': 'fiber optic',
    'dsl': 'dsl',
    'no internet': 'no internet'
})

In [27]:
df['Contract'] = df['Contract'].replace({
    'monthly': 'month-to-month',
    'month-to-month': 'month-to-month',
    '1 year': 'one year',
    'one year': 'one year',
    '2 year': 'two year',
    'two year': 'two year'
})

In [28]:
for col in ['InternetService', 'Contract']:
    print(f"\n{col} Cleaned Unique Values:")
    print(df[col].unique())


InternetService Cleaned Unique Values:
['fiber optic' 'dsl' 'no internet' 'unknown']

Contract Cleaned Unique Values:
['month-to-month' 'one year' 'two year']


In [29]:
#Creating more columns as per requirement for further analysis
# tenure bucket

df['TenureBucket'] = pd.cut(
    df['Tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-12 Months', '13-24 Months', '25-48 Months', '49-72 Months']
)


In [31]:
# segmented customers by monthly spend
df['RevenueSegment'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 40, 80, 300],
    labels=['Low Value', 'Medium Value', 'High Value']
)

In [30]:
#High-risk flag
df['HighRiskCustomer'] = np.where(
    (df['Tenure'] <= 12) &
    (df['Contract'] == 'month-to-month') &
    (df['InternetService'] == 'fiber optic'),
    'Yes',
    'No'
)

In [32]:
df.head()

,CustomerID,Tenure,InternetService,Contract,MonthlyCharges,TotalCharges,PaymentMethod,Churn,TenureBucket,HighRiskCustomer,RevenueSegment
0,C00001,66,fiber optic,month-to-month,140.00,9859.85,electronic check,yes,49-72 Months,No,High Value
1,C00002,17,fiber optic,one year,92.01,1576.77,electronic check,yes,13-24 Months,No,High Value
2,C00003,57,fiber optic,month-to-month,105.94,5161.17,bank transfer,no,49-72 Months,No,High Value
3,C00004,47,fiber optic,one year,41.25,2183.39,mailed check,no,25-48 Months,No,Medium Value
4,C00005,32,dsl,one year,123.04,4145.68,credit card,no,25-48 Months,No,High Value


In [33]:
df.to_csv("cleaned_telecom_churn.csv", index=False)

In [34]:
from google.colab import files
files.download("cleaned_telecom_churn.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>